# SpecDrift-Bench — pilote avec un modèle open-weight servi sur TPU Kaggle (gratuit)

Deux notebooks Kaggle tournent en même temps :

1. **Le serveur (TPU v5e-8)** : le notebook du projet open source [kaggle-tpu-lab](https://github.com/ARahim3/kaggle-tpu-lab) (licence MIT, projet communautaire) sert **Qwen3.8-27B** via une API compatible OpenAI. Sa dernière cellule affiche une **URL** (`https://....trycloudflare.com/v1`) et une **clé API**.
2. **Ce notebook (CPU, sans accélérateur, Internet : On)** : il lance l'agent et le benchmark contre cette URL.

Avant de lancer : dans ce notebook, **Add-ons → Secrets**, crée un secret `TPU_API_KEY` contenant la clé affichée par le serveur. La clé ne doit jamais être écrite en clair dans une cellule.

In [ ]:
!git clone -q https://github.com/roanenicki/specdrift-bench.git
%cd specdrift-bench
!pip install -q mini-swe-agent pyyaml pytest
!python -m pytest -q tests  # doit afficher: 37 passed

## 1. Connexion au serveur

Colle l'URL affichée par le notebook serveur, puis exécute la cellule.

In [ ]:
import os, json, urllib.request
from kaggle_secrets import UserSecretsClient

SERVER_URL = "https://COLLE-ICI.trycloudflare.com/v1"   # <- l'URL affichée par le notebook serveur
os.environ["TPU_API_KEY"] = UserSecretsClient().get_secret("TPU_API_KEY")
os.environ["MSWEA_SILENT_STARTUP"] = "1"
os.environ["MSWEA_COST_TRACKING"] = "ignore_errors"

req = urllib.request.Request(SERVER_URL + "/models", headers={"Authorization": "Bearer " + os.environ["TPU_API_KEY"]})
models = json.load(urllib.request.urlopen(req, timeout=60))
MODEL_ID = models["data"][0]["id"]
MODEL = "openai/" + MODEL_ID
print("serveur joignable, modèle :", MODEL_ID)

## 2. Test rapide : 2 tours (à vérifier avant le pilote)

On veut voir `functional=pass` sur au moins un tour. Sinon, on s'arrête et on analyse avant de consommer du quota.

In [ ]:
import subprocess
subprocess.run(["python", "-m", "specdrift.runner", "--task", "tasks/t01_invlite", "--model", MODEL,
                "--api-base", SERVER_URL, "--api-key-env", "TPU_API_KEY",
                "--condition", "continuous", "--n-turns", "2", "--out", "/kaggle/working/runs_smoke"])

## 3. Pilote : 3 conditions × 2 ordres, 10 tours chacun

Le serveur TPU supporte plusieurs flux en parallèle : on lance les 6 sessions **3 par 3**.
Pour ne pas dépendre du navigateur : **Save Version → Save & Run All (Commit)** sur ce notebook,
pendant que le notebook serveur reste ouvert (une session TPU dure 9 h maximum).

In [ ]:
import subprocess
procs = []
jobs = [(row, cond) for row in [0, 1] for cond in ["continuous", "recall", "fresh"]]
for i, (row, cond) in enumerate(jobs, start=1):
    log = open(f"/kaggle/working/pilot_row{row}_{cond}.log", "w")
    procs.append(subprocess.Popen(
        ["python", "-m", "specdrift.runner", "--task", "tasks/t01_invlite", "--model", MODEL,
         "--api-base", SERVER_URL, "--api-key-env", "TPU_API_KEY", "--condition", cond,
         "--order-row", str(row), "--out", "/kaggle/working/runs"],
        stdout=log, stderr=subprocess.STDOUT))
    print(f"lancé : row {row} / {cond}", flush=True)
    if i % 3 == 0:
        for p in procs[-3:]:
            p.wait()
for p in procs:
    p.wait()
print("pilote terminé")

## 4. Résumé et export

In [ ]:
import json, glob
for f in sorted(glob.glob("/kaggle/working/runs*/**/results.json", recursive=True)):
    r = json.load(open(f)); s = r["summary"]
    print(f.split("working/")[1].rsplit("/", 1)[0],
          "| tours:", len(r["turns"]), "| fonctionnel:", s["functional_success_rate"],
          "| F (tours réussis):", s["mean_F_on_successful_turns"], "| fichiers cassés:", s["turns_with_broken_files"])
!cd /kaggle/working && zip -qr runs.zip runs runs_smoke pilot_*.log 2>/dev/null; ls -lh /kaggle/working/runs.zip